In [2]:
from aegis.schemas import ResearchBundle
from aegis.tools.evidence import load_fixture, lookup_evidence


fixture = load_fixture("football_demo_v1")

assert len(fixture.rows) == 6
print("PASS: six football observations loaded")

lookup = lookup_evidence(
    subject="Player A",
    factor="Scoring",
    dataset_ref="football_demo_v1",
)

assert lookup.reason is None
assert lookup.source is not None
assert len(lookup.evidence) == 2

bundle = ResearchBundle(
    sources=[lookup.source],
    evidence=lookup.evidence,
)

print("PASS: lookup output passes ResearchBundle validation")

for item in bundle.evidence:
    print(item.time_period, item.excerpt)


# Repeating the lookup must preserve evidence identity.
again = lookup_evidence("Player A", "Scoring", "football_demo_v1")

assert [
    item.evidence_id for item in lookup.evidence
] == [
    item.evidence_id for item in again.evidence
]

print("PASS: evidence IDs remain stable")

PASS: six football observations loaded
PASS: lookup output passes ResearchBundle validation
2024 {"goals": 12, "minutes": 1800, "subject": "Player A"}
2025 {"goals": 14, "minutes": 1800, "subject": "Player A"}
PASS: evidence IDs remain stable


In [3]:
from aegis.tools.calculators import (
    normalize,
    per_90,
    rank,
    sensitivity_check,
    weighted_score,
)


# Sum counts and minutes across seasons before calculating per-90.
# Averaging season-level rates directly would weight seasons incorrectly
# when their minutes differ.
subjects = sorted({row.subject for row in fixture.rows})

scoring_rates = {}
creativity_rates = {}

for subject in subjects:
    rows = [row for row in fixture.rows if row.subject == subject]

    minutes = sum(row.minutes for row in rows)
    goals = sum(row.goals for row in rows)
    assists = sum(row.assists for row in rows)

    scoring_rates[subject] = per_90(goals, minutes)

    # For this demonstration only, creativity is proxied by assists/90.
    # This is an explicit modeling choice, not a universal definition.
    creativity_rates[subject] = per_90(assists, minutes)

scoring = normalize(scoring_rates, "higher")
creativity = normalize(creativity_rates, "higher")

factor_scores = {
    subject: {
        "scoring": scoring[subject],
        "creativity": creativity[subject],
    }
    for subject in subjects
}

# Later these preferences will come from the clarified request.
weights = {"scoring": 0.6, "creativity": 0.4}

totals = {
    subject: weighted_score(scores, weights)
    for subject, scores in factor_scores.items()
}

print("Goals per 90:", scoring_rates)
print("Assists per 90:", creativity_rates)
print("Ranking:", rank(totals))

sensitivity = sensitivity_check(factor_scores, weights)

print("Base winners:", sensitivity["base_winners"])
print("Winner changes under perturbation:", sensitivity["winner_changed"])

Goals per 90: {'Player A': 0.65, 'Player B': 0.6096774193548387, 'Player C': 0.5884615384615385}
Assists per 90: {'Player A': 0.35, 'Player B': 0.5516129032258065, 'Player C': 0.31153846153846154}
Ranking: [{'subject': 'Player A', 'score': 0.6640826873385012, 'rank': 1}, {'subject': 'Player B', 'score': 0.6068548387096767, 'rank': 2}, {'subject': 'Player C', 'score': 0.0, 'rank': 3}]
Base winners: ['Player A']
Winner changes under perturbation: False


In [1]:
import math

from aegis.tools.registry import (
    calculator_tool,
    lookup_evidence_tool,
    per_90_tool,
)


# .invoke() receives the same named arguments a future agent will supply.
rate = per_90_tool.invoke({"value": 12, "minutes": 1800})

# Compare floats with a tolerance, never with ==. Binary floats cannot
# store most decimal fractions exactly, so a correct calculation can still
# come back as 0.6000000000000001. math.isclose accepts results that differ
# only by that kind of rounding noise (default relative tolerance 1e-9).
assert math.isclose(rate, 0.6), rate

amount = calculator_tool.invoke(
    {"expression": "(20 * 2) + (2 * 100)"}
)

# Whole numbers are exact here, but use the same rule for every float
# result so a later change to the expression can't break the check.
assert math.isclose(amount, 240.0), amount

gap = lookup_evidence_tool.invoke(
    {
        "subject": "Unknown Player",
        "factor": "Scoring",
        "dataset_ref": "football_demo_v1",
    }
)

# Missing data returns an empty result with a reason instead of raising,
# so an agent can report the gap rather than crash.
assert gap["evidence"] == []
assert gap["reason"]

print("PASS: tools can be invoked through LangChain")
print("Reported evidence gap:", gap["reason"])

PASS: tools can be invoked through LangChain
Reported evidence gap: Subject not found in this fixture: Unknown Player
